In [ ]:
import numpy as np
import sequence.utils.log as log
from sequence.app.teleport_app import TeleportApp
from sequence.topology.node import DQCNode
from sequence.topology.dqc_net_topo import DQCNetTopo
from sequence.kernel.quantum_utils import random_state, verify_same_state_vector
from sequence.constants import MILLISECOND

In [2]:
def teleport(psi: np.array, seed: int = 0) -> np.array:
    """Do a teleportation
    
    Args:
        psi (np.array): the state to be teleported at Alice
        seed (int): the random seed
    Returns:
        np.array: the state received at Bob
    """
    # read the json config file, initialize the topology
    topo = DQCNetTopo("teleport_2node.json")
    tl   = topo.tl  # timeline

    # set up log
    log_filename = 'teleport_2node.log'
    log.set_logger(__name__, tl, log_filename)
    log.set_logger_level('INFO')
    modules = ['generation', 'teleportation', 'teleport_app']
    for module in modules:
        log.track_module(module)

    # get alice and bob node
    nodes = topo.nodes[DQCNetTopo.DQC_NODE]
    alice = next(n for n in nodes if n.name=="alice")
    bob   = next(n for n in nodes if n.name=="bob")

    # 0) set seed for the bsm node
    bsm_nodes = topo.nodes.get(DQCNetTopo.BSM_NODE, [])
    for bsm_node in bsm_nodes:
        bsm_node.set_seed(seed)
    
    # 1) Prepare |ψ> in Alice’s data memory
    data_memo_arr = alice.get_component_by_name(alice.data_memo_arr_name)
    data_memo_arr[0].update_state(psi)

    # 2) Attach the TeleportApp on both nodes
    alice_app = TeleportApp(alice)
    bob_app   = TeleportApp(bob)

    # 3) Kick off teleport
    alice_app.start(
        responder   = bob.name,
        start_t     = 1  * MILLISECOND,
        end_t       = 100 * MILLISECOND,
        memory_size = 1,
        fidelity    = 0.8,
        data_memory_index = 0
    )
    
    # 4) Run the simulation
    tl.init()
    tl.run()

    # 5) simulation finished, read out Bob’s communication qubit state
    timestamp, teleported_qubit = bob_app.results[0]

    return np.array(teleported_qubit)

In [ ]:
def main():

    psi = np.array(random_state())
    seed = np.random.randint(0, 1000)
    
    teleported_psi = teleport(psi, seed)  # run the teleporation
    
    print(f'psi            = {psi}')
    print(f'teleported psi = {teleported_psi}')
    
    assert verify_same_state_vector(teleported_psi, psi), f"teleported state {teleported_psi} != original {psi}"
    
    print('\npsi = teleported_psi --> teleporation success!')
    
    print('\nThe teleport_2node.log:\n')
    with open("teleport_2node.log", "r") as f:
        logs = f.read()
        print(logs)

In [4]:
# please run this cell repeatedly
main()

psi            = [ 0.99543323+0.j         -0.046244  +0.08351159j]
teleported psi = [ 0.99543323+0.j         -0.046244  +0.08351159j]

psi = teleported_psi --> teleporation success!

The teleport_2node.log:

4,505,012,510        INFO    teleportation          alice.teleport.bob.DataMemoryArray[0] bell measurement results: x=0, z=0, remote memory=bob.MemoryArray[0]
5,005,012,510        INFO    teleport_app           bob.TeleportApp: teleport done, state=[ 0.99543323+0.j         -0.046244  +0.08351159j]



In [5]:
import json

distances = [500, 1000, 2000, 5000]

for d in distances:

    # Update distance in JSON
    with open("teleport_2node.json", "r") as f:
        data = json.load(f)

    data["qchannels"][0]["distance"] = d
    data["qchannels"][1]["distance"] = d

    with open("teleport_2node.json", "w") as f:
        json.dump(data, f, indent=4)

    print(f"\n========== Distance = {d} m ==========")

    main()


========== Distance = 500 m ==========
psi            = [ 0.84006812+0.j         -0.46347264+0.28191959j]
teleported psi = [ 0.84006812+0.j         -0.46347264+0.28191959j]

psi = teleported_psi --> teleporation success!

The teleport_2node.log:

16,020,087,510       INFO    teleportation          alice.teleport.bob.DataMemoryArray[0] bell measurement results: x=0, z=1, remote memory=bob.MemoryArray[0]
16,520,087,510       INFO    teleportation          bob.teleport.alice: Z-flip applied on memory bob.MemoryArray[0]
16,520,087,510       INFO    teleport_app           bob.TeleportApp: teleport done, state=[ 0.84006812+0.j         -0.46347264+0.28191959j]


========== Distance = 1000 m ==========
psi            = [0.5457642+0.j         0.8251593-0.14578607j]
teleported psi = [0.5457642+0.j         0.8251593-0.14578607j]

psi = teleported_psi --> teleporation success!

The teleport_2node.log:

4,510,012,510        INFO    teleportation          alice.teleport.bob.DataMemoryArray[0] bell 